# MuseTalk 1.5
Official: https://github.com/TMElyralab/MuseTalk . Uses a separate Python 3.10 environment because upstream pins older Torch and MMLab. First installation may compile MMCV and can fail on newer GPU runtimes.
Run cells in order on a fresh GPU runtime. Use a single visible face and an isolated vocal stem. Input offsets are independent; final output contains test audio, not the full music mix. Not GPU-tested.

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path
from google.colab import drive, files
from IPython.display import Video, display
def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)
run('nvidia-smi')
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/Alina_LipSync')
root.mkdir(parents=True, exist_ok=True)
run('apt-get','-qq','update')
run('apt-get','-qq','install','-y','ffmpeg')


In [ ]:
repo=Path('/content/MuseTalk')
if not repo.exists(): run('git','clone','https://github.com/TMElyralab/MuseTalk.git',repo)
run(sys.executable,'-m','pip','install','uv')
env=Path('/content/musetalk_env')
if not env.exists(): run('uv','venv','--python','3.10','--seed',env)
py=env/'bin/python'
run('uv','pip','install','--python',py,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118')
run(py,'-m','pip','install','-r',repo/'requirements.txt','openmim','huggingface_hub')
run(env/'bin/mim','install','mmengine','mmcv==2.0.1','mmdet==3.1.0','mmpose==1.1.0')
weights=root/'MuseTalk_models'; weights.mkdir(exist_ok=True)
if not (repo/'models').exists(): (repo/'models').symlink_to(weights)
os.environ['PATH']=str(env/'bin')+':'+os.environ['PATH']
run('bash','download_weights.sh',cwd=repo)
run(py,'-m','scripts.inference','--help',cwd=repo)

In [ ]:
VIDEO_PATH = '' #@param {type:"string"}
AUDIO_PATH = '' #@param {type:"string"}
VIDEO_START = 0.0 #@param {type:"number"}
AUDIO_START = 0.0 #@param {type:"number"}
DURATION = 8.0 #@param {type:"number"}
def choose(path):
    if path:
        p=Path(path).resolve()
    else:
        uploaded=files.upload()
        assert len(uploaded)==1, 'Upload exactly one file'
        p=Path(next(iter(uploaded))).resolve()
    assert p.is_file(), p
    return p
assert DURATION>0 and VIDEO_START>=0 and AUDIO_START>=0
video=choose(VIDEO_PATH)
audio=choose(AUDIO_PATH)
work=Path('/content/alina_inputs'); work.mkdir(exist_ok=True)
v=work/'video.mp4'; a=work/'vocals.wav'
run('ffmpeg','-y','-ss',VIDEO_START,'-i',video,'-t',DURATION,'-an','-vf','fps=25','-c:v','libx264','-pix_fmt','yuv420p',v)
run('ffmpeg','-y','-ss',AUDIO_START,'-i',audio,'-t',DURATION,'-vn','-ac','1','-ar','16000',a)


In [ ]:
import json
cfg=work/'muse_config.yaml'
cfg.write_text(json.dumps({'task_0':{'video_path':str(v),'audio_path':str(a)}}))
result=Path('/content/muse_results'); result.mkdir(exist_ok=True)
before={p:p.stat().st_mtime_ns for p in result.rglob('*.mp4')}
t=time.perf_counter()
run(py,'-m','scripts.inference','--inference_config',cfg,'--result_dir',result,'--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--version','v15','--ffmpeg_path','/usr/bin',cwd=repo)
candidates=[p for p in result.rglob('*.mp4') if p not in before or p.stat().st_mtime_ns!=before[p]]
assert candidates,'No fresh output found'
out=root/'musetalk_output.mp4'
shutil.copy2(max(candidates,key=lambda p:p.stat().st_mtime_ns),out)
print('Seconds:',time.perf_counter()-t)
assert out.is_file(), out
print('Saved:',out)
display(Video(str(out),embed=True))
# Uncomment to download:
# files.download(str(out))